In [5]:
from data_model.DataCleaner import *
from dl_client import DatalakeClient
from data_model.manage_excel_support_file import *
from data_model.MergerTools import *
import pandas as pd
import os

client = DatalakeClient()
mergeTools = MergerTools()

In [10]:
search = client.query_files(
    query={'custom.level' : 'merged_comb'})

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True)

print(len(zip_files))

20


In [12]:
file_names = list(zip_files.keys())
print(file_names)

['merge_FSVERSION_4-3_METHOD_CSF_elecsys_METHOD_PET_FBB.csv', 'merge_FSVERSION_4-3_METHOD_CSF_lumipulse_METHOD_PET_FBB.csv', 'merge_FSVERSION_4-3_METHOD_CSF_elecsys_METHOD_PET_FBP.csv', 'merge_FSVERSION_4-3_METHOD_CSF_lumipulse_METHOD_PET_FBP.csv', 'merge_FSVERSION_4-4_METHOD_CSF_elecsys_METHOD_PET_FBB.csv', 'merge_FSVERSION_4-4_METHOD_CSF_lumipulse_METHOD_PET_FBB.csv', 'merge_FSVERSION_4-4_METHOD_CSF_elecsys_METHOD_PET_FBP.csv', 'merge_FSVERSION_4-4_METHOD_CSF_lumipulse_METHOD_PET_FBP.csv', 'merge_FSVERSION_5-1_METHOD_CSF_elecsys_METHOD_PET_FBB.csv', 'merge_FSVERSION_5-1_METHOD_CSF_lumipulse_METHOD_PET_FBB.csv', 'merge_FSVERSION_5-1_METHOD_CSF_elecsys_METHOD_PET_FBP.csv', 'merge_FSVERSION_5-1_METHOD_CSF_lumipulse_METHOD_PET_FBP.csv', 'merge_FSVERSION_6-0_METHOD_CSF_elecsys_METHOD_PET_FBB.csv', 'merge_FSVERSION_6-0_METHOD_CSF_lumipulse_METHOD_PET_FBB.csv', 'merge_FSVERSION_6-0_METHOD_CSF_elecsys_METHOD_PET_FBP.csv', 'merge_FSVERSION_6-0_METHOD_CSF_lumipulse_METHOD_PET_FBP.csv', 'merge_

In [13]:
dfs = {}
df_names = {}
df_code = []
idx = 0
for file_name, df_raw in zip_files.items():
    print('\n### ', file_name)
    df_copy = df_raw.copy(deep=True)
    df_copy['EXAMDATE'] = pd.to_datetime(df_copy['EXAMDATE'])
    df_copy = df_copy.sort_values(by=['RID', 'EXAMDATE']).reset_index(drop=True)
    duplicati_ridage = df_copy.groupby(['RID', 'AGE']).apply(lambda x: x.index.tolist()).loc[lambda x: x.str.len() > 1]  
    # aggiornamento liste e dizionari
    dfs[f"df_{idx}"] = df_copy  
    df_names[f"df_{idx}"] = file_name 
    df_code.append(f"df_{idx}")
    # definizione variabile df
    globals()[f"df_{idx}"] = df_copy
    print(idx, '--->', file_name, '\n', duplicati_ridage)
    idx += 1


###  merge_FSVERSION_4-3_METHOD_CSF_elecsys_METHOD_PET_FBB.csv
0 ---> merge_FSVERSION_4-3_METHOD_CSF_elecsys_METHOD_PET_FBB.csv 
 RID     AGE 
54.0    81.0        [264, 265]
55.0    76.0        [271, 273]
60.0    70.0        [330, 331]
61.0    77.0        [336, 337]
78.0    76.0        [439, 440]
                     ...      
6981.0  66.0    [16007, 16008]
6989.0  68.0    [16017, 16018]
7012.0  57.0    [16084, 16085]
7018.0  63.0    [16102, 16103]
7055.0  72.0    [16178, 16179]
Length: 233, dtype: object

###  merge_FSVERSION_4-3_METHOD_CSF_lumipulse_METHOD_PET_FBB.csv
1 ---> merge_FSVERSION_4-3_METHOD_CSF_lumipulse_METHOD_PET_FBB.csv 
 RID     AGE 
54.0    81.0        [262, 263]
55.0    76.0        [269, 271]
60.0    70.0        [328, 329]
61.0    77.0        [334, 335]
78.0    76.0        [435, 436]
                     ...      
6981.0  66.0    [15916, 15917]
6989.0  68.0    [15926, 15927]
7012.0  57.0    [15993, 15994]
7018.0  63.0    [16011, 16012]
7055.0  72.0    [16086, 16087]

In [31]:
import json
from collections import defaultdict

def recalculate_age(df, df_name="unknown"):
    """
    Ricalcola AGE per ogni soggetto basandosi sulla prima visita con AGE valido.
    
    Logica:
    - Per ogni RID, trova la prima riga con AGE valido (ancora)
    - Calcola AGE per tutte le altre righe: AGE_ancora + delta_temporale
    - Se EXAMDATE mancante, usa VISIT_MONTH come fallback
    - Se anche VISIT_MONTH mancante, AGE = NaN
    
    Returns:
        df: DataFrame con AGE ricalcolato
        warnings: dict con warnings per df_name
    """
    df = df.copy()
    warnings = defaultdict(list)
    
    for rid in df['RID'].unique():
        mask_rid = df['RID'] == rid
        idx_rid = df[mask_rid].index.tolist()
        
        # Trova prima riga con AGE valido
        anchor_idx = None
        for idx in idx_rid:
            if pd.notna(df.loc[idx, 'AGE']):
                anchor_idx = idx
                break
        
        # Se nessuna riga ha AGE valido, skip (tutte rimangono NaN)
        if anchor_idx is None:
            warnings["no_valid_age_for_subject"].append({
                "RID": rid,
                "indices": idx_rid
            })
            continue
        
        # Warning se la prima visita non ha AGE valido
        if anchor_idx != idx_rid[0]:
            warnings["first_visit_age_nan"].append({
                "RID": rid,
                "first_idx": idx_rid[0],
                "anchor_idx": anchor_idx
            })
        
        anchor_age = df.loc[anchor_idx, 'AGE']
        anchor_examdate = df.loc[anchor_idx, 'EXAMDATE']
        anchor_visit_month = df.loc[anchor_idx, 'VISIT_MONTH']
        
        # Ricalcola AGE per tutte le righe del soggetto
        for idx in idx_rid:
            if idx == anchor_idx:
                continue  # Mantieni AGE ancora
            
            current_examdate = df.loc[idx, 'EXAMDATE']
            current_visit_month = df.loc[idx, 'VISIT_MONTH']
            
            # Prova con EXAMDATE
            if pd.notna(current_examdate) and pd.notna(anchor_examdate):
                delta_days = (current_examdate - anchor_examdate).days
                delta_years = delta_days / 365.25
                df.loc[idx, 'AGE'] = round(anchor_age + delta_years, 3)
            
            # Fallback: usa VISIT_MONTH
            elif pd.notna(current_visit_month) and pd.notna(anchor_visit_month):
                delta_months = current_visit_month - anchor_visit_month
                delta_years = delta_months / 12
                df.loc[idx, 'AGE'] = round(anchor_age + delta_years, 3)
                warnings["missing_examdate"].append({
                    "RID": rid,
                    "idx": idx,
                    "used_visit_month": True
                })
            
            # Nessuna info temporale
            else:
                df.loc[idx, 'AGE'] = np.nan
                warnings["no_temporal_info"].append({
                    "RID": rid,
                    "idx": idx
                })
    
    return df, dict(warnings)

def count_duplicate_rid_age(df):
    """Conta il numero di coppie (RID, AGE) duplicate."""
    duplicates = df.groupby(['RID', 'AGE']).size()
    return (duplicates > 1).sum()

def get_duplicate_rid_age_pairs(df):
    """Restituisce le coppie (RID, AGE) duplicate."""
    duplicates = df.groupby(['RID', 'AGE']).size()
    return set(duplicates[duplicates > 1].index.tolist())

print("Funzioni definite.")

Funzioni definite.


In [32]:
# Applica recalculate_age a tutti i 20 dataframe
# Salva anche i conteggi duplicati PRIMA del ricalcolo
all_warnings = {}
duplicates_before = {}
duplicates_pairs_before = {}

for df_key in df_code:
    df_original = dfs[df_key]
    file_name = df_names[df_key]
    
    # Conta duplicati PRIMA
    duplicates_before[df_key] = count_duplicate_rid_age(df_original)
    duplicates_pairs_before[df_key] = get_duplicate_rid_age_pairs(df_original)
    
    print(f"\nProcessando {df_key} ({file_name})...")
    print(f"  Duplicati (RID, AGE) PRIMA: {duplicates_before[df_key]}")
    
    # Ricalcola età
    df_fixed, warnings = recalculate_age(df_original, df_name=file_name)
    
    # Aggiorna dizionario e variabile globale
    dfs[df_key] = df_fixed
    globals()[df_key] = df_fixed
    
    # Salva warnings
    if warnings:
        all_warnings[file_name] = warnings
        print(f"  Warnings: {', '.join([f'{k}: {len(v)}' for k, v in warnings.items()])}")
    else:
        print(f"  Nessun warning")

print(f"\n{'='*50}")
print(f"Completato! Processati {len(df_code)} dataframe.")
print(f"Dataframe con warnings: {len(all_warnings)}")


Processando df_0 (merge_FSVERSION_4-3_METHOD_CSF_elecsys_METHOD_PET_FBB.csv)...
  Duplicati (RID, AGE) PRIMA: 233
  Warnings: first_visit_age_nan: 22, no_valid_age_for_subject: 85

Processando df_1 (merge_FSVERSION_4-3_METHOD_CSF_lumipulse_METHOD_PET_FBB.csv)...
  Duplicati (RID, AGE) PRIMA: 233
  Warnings: first_visit_age_nan: 22, no_valid_age_for_subject: 85

Processando df_2 (merge_FSVERSION_4-3_METHOD_CSF_elecsys_METHOD_PET_FBP.csv)...
  Duplicati (RID, AGE) PRIMA: 233
  Warnings: first_visit_age_nan: 23, no_valid_age_for_subject: 85

Processando df_3 (merge_FSVERSION_4-3_METHOD_CSF_lumipulse_METHOD_PET_FBP.csv)...
  Duplicati (RID, AGE) PRIMA: 233
  Warnings: first_visit_age_nan: 23, no_valid_age_for_subject: 85

Processando df_4 (merge_FSVERSION_4-4_METHOD_CSF_elecsys_METHOD_PET_FBB.csv)...
  Duplicati (RID, AGE) PRIMA: 233
  Warnings: first_visit_age_nan: 21, no_valid_age_for_subject: 85

Processando df_5 (merge_FSVERSION_4-4_METHOD_CSF_lumipulse_METHOD_PET_FBB.csv)...
  Duplic

In [33]:
# Salva warnings in JSON
warnings_file = "age_recalculation_warnings.json"

with open(warnings_file, 'w', encoding='utf-8') as f:
    json.dump(all_warnings, f, indent=2, default=str)

print(f"Warnings salvati in: {warnings_file}")

# Riepilogo warnings
total_warnings = sum(len(w) for warns in all_warnings.values() for w in warns.values())
print(f"Totale warnings: {total_warnings}")

Warnings salvati in: age_recalculation_warnings.json
Totale warnings: 2157


In [34]:
# Riepilogo aggregato: confronto PRIMA vs DOPO
print("=" * 70)
print("RIEPILOGO DUPLICATI (RID, AGE)")
print("=" * 70)
print(f"{'DataFrame':<10} {'File':<55} {'PRIMA':>6} {'DOPO':>6} {'NUOVI':>6}")
print("-" * 70)

total_before = 0
total_after = 0
total_new = 0
new_duplicates_detail = {}

for df_key in df_code:
    file_name = df_names[df_key]
    short_name = file_name.replace("merge_", "").replace(".csv", "")[:50]
    
    before = duplicates_before[df_key]
    after = count_duplicate_rid_age(dfs[df_key])
    
    # Trova nuovi duplicati (coppie che non c'erano prima)
    pairs_after = get_duplicate_rid_age_pairs(dfs[df_key])
    pairs_before = duplicates_pairs_before[df_key]
    new_pairs = pairs_after - pairs_before
    
    if new_pairs:
        new_duplicates_detail[file_name] = list(new_pairs)
    
    total_before += before
    total_after += after
    total_new += len(new_pairs)
    
    status = "⚠️" if new_pairs else "✓"
    print(f"{df_key:<10} {short_name:<55} {before:>6} {after:>6} {len(new_pairs):>6} {status}")

print("-" * 70)
print(f"{'TOTALE':<66} {total_before:>6} {total_after:>6} {total_new:>6}")
print("=" * 70)

if new_duplicates_detail:
    print("\n⚠️ ATTENZIONE: Nuovi duplicati creati!")
    for file_name, pairs in new_duplicates_detail.items():
        print(f"\n  {file_name}:")
        for rid, age in pairs[:5]:  # Mostra max 5
            print(f"    RID={rid}, AGE={age}")
        if len(pairs) > 5:
            print(f"    ... e altri {len(pairs) - 5}")
else:
    print("\n✓ Nessun nuovo duplicato creato.")

RIEPILOGO DUPLICATI (RID, AGE)
DataFrame  File                                                     PRIMA   DOPO  NUOVI
----------------------------------------------------------------------
df_0       FSVERSION_4-3_METHOD_CSF_elecsys_METHOD_PET_FBB            233      0      0 ✓
df_1       FSVERSION_4-3_METHOD_CSF_lumipulse_METHOD_PET_FBB          233      0      0 ✓
df_2       FSVERSION_4-3_METHOD_CSF_elecsys_METHOD_PET_FBP            233      0      0 ✓
df_3       FSVERSION_4-3_METHOD_CSF_lumipulse_METHOD_PET_FBP          233      0      0 ✓
df_4       FSVERSION_4-4_METHOD_CSF_elecsys_METHOD_PET_FBB            233      0      0 ✓
df_5       FSVERSION_4-4_METHOD_CSF_lumipulse_METHOD_PET_FBB          233      0      0 ✓
df_6       FSVERSION_4-4_METHOD_CSF_elecsys_METHOD_PET_FBP            233      0      0 ✓
df_7       FSVERSION_4-4_METHOD_CSF_lumipulse_METHOD_PET_FBP          233      0      0 ✓
df_8       FSVERSION_5-1_METHOD_CSF_elecsys_METHOD_PET_FBB            233      0      0 ✓


In [ ]:
# Colonne da controllare per valori validi
COLS_TO_CHECK = [
    'Ventricles%ICV', 'Hippocampus%ICV', 'Entorhinal%ICV', 'Fusiform%ICV', 'MidTemp%ICV',
    'ADAS11', 'ADAS13', 'CDRGLOB', 'CDRSB', 'FAQ', 'MMSE', 'MOCA', 'RAVLT_immediate',
    'AB40_CSF', 'AB4240_CSF', 'AB42_CSF', 'PT181_AB42_CSF', 'PT181_CSF', 'TTAU_AB42_CSF', 'TTAU_CSF',
    'AMY_CENTILOIDS', 'CTX_ENTORHINAL_SUVR', 'CTX_FUSIFORM_SUVR', 'CTX_INFERIORPARIETAL_SUVR',
    'CTX_LATERALOCCIPITAL_SUVR', 'CTX_MIDDLETEMPORAL_SUVR', 'CTX_PARAHIPPOCAMPAL_SUVR',
    'ENTORHINAL_SUVR', 'INFERIOR_TEMPORAL_SUVR', 'SUMMARY_SUVR', 'TAU_METAROI', 'TRACER',
    'APOE', 'APOE_4', 'DX/CN', 'DX/Dementia', 'DX/MCI', 'EDUCAT',
    'ETHNICITY/latino', 'ETHNICITY/not_latino', 'GENDER/female', 'GENDER/male',
    'MARRY/divorced', 'MARRY/married', 'MARRY/single', 'MARRY/widowed',
    'RACE/Asian', 'RACE/Black', 'RACE/Mixed', 'RACE/Native_american', 'RACE/White'
]

def remove_rows_with_few_values(df, cols_to_check, min_valid=2):
    """
    Rimuove righe con meno di min_valid valori non-NaN nelle colonne specificate.
    
    Args:
        df: DataFrame
        cols_to_check: lista colonne da controllare
        min_valid: minimo numero di valori validi richiesti (default 2, quindi elimina se <= 1)
    
    Returns:
        df_cleaned, n_removed
    """
    # Filtra solo colonne esistenti nel df
    cols_present = [c for c in cols_to_check if c in df.columns]
    
    # Conta valori non-NaN per riga
    valid_counts = df[cols_present].notna().sum(axis=1)
    
    # Maschera: righe da mantenere (valori validi >= min_valid)
    mask_keep = valid_counts >= min_valid
    
    n_removed = (~mask_keep).sum()
    df_cleaned = df[mask_keep].reset_index(drop=True)
    
    return df_cleaned, n_removed

# Applica a tutti i df
print("=" * 60)
print("RIMOZIONE RIGHE CON <= 1 VALORE VALIDO")
print("=" * 60)
print(f"Colonne controllate: {len(COLS_TO_CHECK)}")
print("-" * 60)
print(f"{'DataFrame':<10} {'Righe prima':>15} {'Righe dopo':>15} {'Rimosse':>10}")
print("-" * 60)

total_before = 0
total_after = 0
total_removed = 0

for df_key in df_code:
    df_original = dfs[df_key]
    rows_before = len(df_original)
    
    df_cleaned, n_removed = remove_rows_with_few_values(df_original, COLS_TO_CHECK, min_valid=2)
    
    # Aggiorna dizionario e variabile globale
    dfs[df_key] = df_cleaned
    globals()[df_key] = df_cleaned
    
    rows_after = len(df_cleaned)
    total_before += rows_before
    total_after += rows_after
    total_removed += n_removed
    
    print(f"{df_key:<10} {rows_before:>15} {rows_after:>15} {n_removed:>10}")

print("-" * 60)
print(f"{'TOTALE':<10} {total_before:>15} {total_after:>15} {total_removed:>10}")
print("=" * 60)

In [41]:
df_0.loc[[6736], ['RID','VISIT_MONTH', 'EXAMDATE', 'AGE', 'AGE_bl']]

,RID,VISIT_MONTH,EXAMDATE,AGE,AGE_bl
6736,2016.0,0.0,2010-06-22,NaN,NaN


In [46]:
df_0[df_0['RID']==2053]

,RID,COHORT,VISCODE,VISIT_MONTH,EXAMDATE,FLDSTRENG,FSVERSION,IMAGEUID,update_stamp,Ventricles%ICV,...,MARRY/divorced,MARRY/married,MARRY/single,MARRY/widowed,RACE/Asian,RACE/Black,RACE/Mixed,RACE/Native_american,RACE/White,Tprofile
6838,2053.0,ADNIGO,sc,0.0,2010-08-12,NaN,NaN,NaN,2024-08-22 07:18:34.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [47]:
df_0.loc[df_0['RID']==2053].dropna(axis=1, how='all').columns


Index(['RID', 'COHORT', 'VISCODE', 'VISIT_MONTH', 'EXAMDATE', 'update_stamp',
       'MMSE'],
      dtype='object')